# 01 — Holon Basics

Every holon is an **IRI** whose associated **named graphs** live in an
RDF dataset. The four-graph model organizes those graphs into four
roles: interior, boundary, projection, context.

This notebook covers:

1. Creating a holon and adding multiple interior graphs
2. Declaring a SHACL boundary
3. Running a union-of-interiors SPARQL query
4. Membrane validation

All operations go through `HolonicDataset` against the default
`RdflibBackend` (in-memory).


## Setup

Fresh dataset with the CGA ontology loaded.


In [ ]:
from holonic import HolonicDataset

ds = HolonicDataset()  # rdflib in-memory backend, ontology auto-loaded
ds.summary()


## Create a holon

A holon is declared with `add_holon(iri, label)`. The IRI is the
identity anchor for all four layers.


In [ ]:
ds.add_holon("urn:holon:vancouver", "Vancouver")


## Add multi-source interiors

A holon may have **multiple interior graphs**. Each carries a distinct
slice of what the holon knows about itself — for instance, data from
different upstream systems.

Reads across interiors are done as a union: any SPARQL query spanning
two `GRAPH ?g { ... }` patterns finds triples regardless of which
interior graph they came from.


In [ ]:
ds.add_interior("urn:holon:vancouver", '''
    @prefix muni: <urn:municipal:> .
    <urn:city:vancouver> a muni:CityRecord ;
        muni:officialName "City of Vancouver" ;
        muni:censusPop 675218 ;
        muni:lat 49.2827 ;
        muni:lon -123.1207 .
''', graph_iri="urn:holon:vancouver/interior/municipal")

ds.add_interior("urn:holon:vancouver", '''
    @prefix census: <urn:census:> .
    <urn:city:vancouver> census:year 2021 ;
        census:province "British Columbia" .
''', graph_iri="urn:holon:vancouver/interior/census")


## Declare a boundary

A boundary graph holds SHACL shapes that express what's allowed
inside the holon. The shapes are stored as triples, not as Python
objects — they're queryable like any other RDF.


In [ ]:
ds.add_boundary("urn:holon:vancouver", '''
    @prefix muni: <urn:municipal:> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .

    <urn:shapes:CityShape> a sh:NodeShape ;
        sh:targetClass muni:CityRecord ;
        sh:property [
            sh:path muni:officialName ;
            sh:minCount 1 ;
            sh:datatype xsd:string ;
            sh:severity sh:Violation
        ] ;
        sh:property [
            sh:path muni:censusPop ;
            sh:minCount 1 ;
            sh:datatype xsd:integer ;
            sh:severity sh:Violation
        ] .
''')


## Union SPARQL across interior graphs

Values that live in separate interior graphs join naturally:


In [ ]:
rows = ds.query('''
    PREFIX muni: <urn:municipal:>
    PREFIX census: <urn:census:>
    SELECT ?name ?pop ?year
    WHERE {
        GRAPH ?g1 {
            <urn:city:vancouver> muni:officialName ?name ;
                muni:censusPop ?pop .
        }
        GRAPH ?g2 {
            <urn:city:vancouver> census:year ?year .
        }
    }
''')
rows


## Membrane validation

`validate_membrane(iri)` runs pyshacl against the union of the
holon's interior graphs using the union of its boundary graphs. The
return value indicates overall health: `Intact` / `Weakened` /
`Compromised`.


In [ ]:
result = ds.validate_membrane("urn:holon:vancouver")
print(f"health: {result.health}")
print(f"violations: {len(result.violations)}")
print(f"warnings:   {len(result.warnings)}")


## Inspect the holarchy

`summary()` prints backend + holon + portal + graph counts. For
programmatic inspection, use `list_holons()` or `get_holon_detail(iri)`.


In [ ]:
print(ds.summary())


## Where to go next

- `02_portal_traversal` — connect two holons with a CONSTRUCT portal
- `09_console_views` — lightweight queries for web-UI consumption
- `07_graph_metadata` — triple counts and class inventories per graph
